In [ ]:
# CELL 0 --- IMPORTS
# All imports consolidated here (previously scattered across Cells 2/5/6).
# Single source for environment + logging configuration too. 
     
# Standard library
import json
import logging
import re
from datetime import datetime
from pathlib import Path
 
# Third-party
import duckdb
import joblib
import numpy as np
import pandas as pd
from scipy.stats import median_abs_deviation
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
 
# Logging — configured exactly once, here. (Calling `basicConfig` later in
# other cells was a no-op, which masked configuration drift.)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [ ]:
# ============================================================
# CELL 1 — MASTER FEATURE SCHEMA (Single Source of Truth)
# Run this cell first before anything else
# ============================================================
LOGON_FEATURES = [
    'total_events_buckets',    # renamed from 'total_events'
                               #  Count of distinct (user, pc, activity, 5-min-bucket)
                               # tuples — i.e. deduplicated event buckets, not raw events.
    'off_hours_ratio',
    'unique_systems_accessed',
    'failed_login_ratio',
    'active_orphan_sessions'
]

HTTP_FEATURES = [
    'url_visits',
    'upload_event_count',      # renamed from 'bytes_uploaded'
                               # Count of upload/post events, NOT bytes.
                               # CERT r4.2 http.csv has no size column;
                               # real bytes would require a different feed.
    'keyword_match_indicator', 
                               # renamed from 'malicious_domain_hits'
                               # Hardcoded 2-string heuristic — see Cell 3 comment.
                               # PRODUCTION: replace with URLhaus / OpenPhish lookup.
    'unique_external_domains',
    'after_hours_browsing'
]

EMAIL_FEATURES = [
    'emails_sent',
    'external_recipients',
    'after_hours_emails',
    'attachments_sent',
    'large_attachment_count'
]

MASTER_FEATURE_SCHEMA = LOGON_FEATURES + HTTP_FEATURES + EMAIL_FEATURES


def align_to_master_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures DataFrame always conforms to master feature schema.
    Missing columns (http, email) filled with NaN until those
    sources are pulled. Enforces consistent column ordering.
    Does NOT mutate the caller's DataFrame — works on a copy.
    """
    df = df.copy()                      # Don't surprise the caller
    for col in MASTER_FEATURE_SCHEMA:
        if col not in df.columns:
            df[col] = np.nan
    return df[MASTER_FEATURE_SCHEMA]


In [ ]:
# ============================================================
# CELL 2 — LOGON EXTRACTION
# ============================================================
# Date pattern used by all three extraction functions
_DATE_PATTERN = re.compile(r'^\d{4}-\d{2}-\d{2}$')


def _validate_extraction_inputs(
    user_input_path: str, 
    start_date: str = None, 
    end_date: str = None, 
    dev_mode: bool = False
) -> Path:
    """
    Shared input validation for all three extraction functions.
 
    Returns the resolved target file path on success.
    Raises ValueError / FileNotFoundError on any violation.
 
    Factored out so the three SQL functions don't each duplicate ~30 lines
    of identical validation (DRY).
    """
    logging.info("Initiating Feature Extraction Pipeline...")

    # ============================================================
    # 1. SECURITY SANDBOX & PATH VALIDATION
    # ============================================================
    if dev_mode:
        logging.warning("Running in DEV_MODE. Strict security sandbox is DISABLED.")
        target_file: Path = Path(user_input_path).resolve()

        if not target_file.exists():
            logging.error(f"DEV ERROR: Target file not found at {target_file}")
            raise FileNotFoundError(f"Cannot find file at {target_file}")
    else:
        # PROD MODE: Enforce the Sandbox (Using a local relative 'data' folder)
        safe_directory: Path = Path("data").resolve()
        target_file: Path = (safe_directory / user_input_path).resolve()

        if not target_file.is_relative_to(safe_directory):
            logging.error(f"SECURITY ALERT: Directory Traversal Attempt Detected: {user_input_path}")
            raise ValueError("Access Denied: Invalid file path.")

    if target_file.suffix != '.csv':
        logging.error(f"SECURITY ALERT: Invalid file type attempted: {target_file.suffix}")
        raise ValueError("Access Denied: Only .csv files are permitted.")

    # CRITICAL WINDOWS FIX: Convert backslashes (\) to forward slashes (/)
    # so DuckDB doesn't treat them as SQL escape characters.
    target_file_str = target_file.as_posix()
    
    if "'" in target_file_str:
        raise ValueError("File path contains illegal character. Single quotes are not permitted.")

    # ── DATE PARAMETER VALIDATION (SQL injection prevention) ────────────
    # start_date / end_date are interpolated into a DuckDB query string.
    # We reject anything that doesn't match YYYY-MM-DD exactly.
    if start_date is not None and not _DATE_PATTERN.match(start_date):
        raise ValueError(f"start_date '{start_date}' is not YYYY-MM-DD. Example: '2010-01-01'")
    if end_date is not None and not _DATE_PATTERN.match(end_date):
        raise ValueError(f"end_date '{end_date}' is not YYYY-MM-DD. Example: '2011-04-01'")
 
    return target_file

def _build_output_path(output_prefix: str) -> Path:
    """Creates ./features/ if needed and returns a timestamped output path."""
    output_dir = Path("features").resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = output_dir / f"{output_prefix}_{timestamp}.parquet"
    if "'" in output_file.as_posix():
        raise ValueError("Output path contains illegal character (single quote).")
    return output_file


def _build_date_filter_sql(start_date: str | None, end_date: str | None) -> str:
    """Builds the date filter fragment used in all three extraction queries."""
    sql_fragment = ""
    if start_date:
        sql_fragment += (
            f" AND strptime(date, '%m/%d/%Y %H:%M:%S') "
            f">= TIMESTAMP '{start_date}'"
        )
    if end_date:
        sql_fragment += (
            f" AND strptime(date, '%m/%d/%Y %H:%M:%S') "
            f"< TIMESTAMP '{end_date}'"
        )
    return sql_fragment


def extract_logon_features(
    user_input_path: str,
    start_date: str | None = None,
    end_date: str | None = None,
    output_prefix: str = "logon_features",
    dev_mode: bool = False,
) -> str:
    """
    Extracts daily behavioural features from raw logon logs.
 
    Output schema (DataFrame columns after this function):
        total_event_buckets       — count of (user, pc, activity, 5-min-bucket) tuples
        off_hours_ratio           — fraction of events outside 06:00–18:00 Mon–Fri
        unique_systems_accessed   — distinct PCs per day
        failed_login_ratio        — failed events / total event buckets
        active_orphan_sessions    — (logons - logoffs) per day
    """

    logging.info("Initiating Logon Feature Extraction Pipeline...")
 
    target_file = _validate_extraction_inputs(user_input_path, start_date, end_date, dev_mode)
    output_file = _build_output_path(output_prefix)
    date_filter_sql = _build_date_filter_sql(start_date, end_date)
 
    target_file_str = target_file.as_posix()
    output_file_str = output_file.as_posix()

    # ── SQL ─────────────────────────────────────────────────────────────
    # NOTE on `total_event_buckets`:
    # The inner CTE GROUPs by (user, pc, activity, session_time), which
    # deduplicates events within the same 5-minute bucket. COUNT(*) outside
    # the CTE therefore counts BUCKETS, not raw events. Renamed from
    # `total_events` to reflect this honestly.
    query: str = f"""
    COPY (
        -- ── STEP 1: Deduplication & Sessionization ──────────────────────
        WITH SessionizedLogs AS (
            SELECT
                user,
                pc,          
                activity,    
                TIME_BUCKET(
                    INTERVAL '5 minutes',
                    strptime(date, '%m/%d/%Y %H:%M:%S')
                ) AS session_time
            FROM read_csv_auto('{target_file_str}')
            WHERE user != 'SYSTEM'
                -- [FIX 2] Added 'Logoff' to the filter to track session closure
                AND (activity IN ('Logon', 'Logoff') OR activity ILIKE '%fail%')
                {date_filter_sql} -- [NEW] Injected Time Split Logic
            GROUP BY user, pc, activity, session_time
    )

        -- ── STEP 2: Per-User DAILY Feature Aggregation ───────────────────
        SELECT
            user,
            CAST(session_time AS DATE) AS activity_date, -- [FIX 1] Temporal Grouping

            COUNT(*) AS total_event_buckets,     -- honest name

            SUM(
                CASE
                    WHEN DAYOFWEEK(session_time) IN (0, 6) THEN 1   -- Weekend
                    WHEN EXTRACT(hour FROM session_time) < 6
                      OR EXTRACT(hour FROM session_time) >= 18 THEN 1 -- Night hours
                    ELSE 0
                END
            ) * 1.0 / NULLIF(COUNT(*), 0) AS off_hours_ratio,

            COUNT(DISTINCT pc) AS unique_systems_accessed,

            SUM(
                CASE WHEN activity ILIKE '%fail%' THEN 1 ELSE 0 END
            ) * 1.0 / NULLIF(COUNT(*), 0) AS failed_login_ratio,

            -- [FIX 2] Orphan Session Indicator (Logons minus Logoffs)
            -- A high positive number means they are leaving sessions open 
            -- (Lateral movement or scripted activity)
            SUM(CASE WHEN activity = 'Logon' THEN 1 ELSE 0 END) -
            SUM(CASE WHEN activity = 'Logoff' THEN 1 ELSE 0 END) AS active_orphan_sessions

        FROM SessionizedLogs
        GROUP BY user, CAST(session_time AS DATE) -- [FIX 1] Group by User AND Date

    ) TO '{output_file_str}' (FORMAT PARQUET);
    """

    # ============================================================
    # 4. EXECUTION BLOCK
    # ============================================================
    try:
        duckdb.sql(query)
    except Exception as e:
        logging.error(f"[extract_logon_features] DuckDB execution failed. Reason: {e}")
        raise

    # ============================================================
    # 5. POST-EXECUTION VALIDATION + SCHEMA ALIGNMENT
    # ============================================================
    df_check = pd.read_parquet(output_file)

    if len(df_check) == 0:
        raise RuntimeError("Pipeline produced empty output. Check WHERE filters.")

    # [FIX 1 Update] The index is now a MultiIndex: User AND Date.
    # This is required for time-series anomaly detection.
    df_aligned = align_to_master_schema(df_check.set_index(['user', 'activity_date']))

    df_aligned.to_parquet(output_file, index=True)

    logging.info(
        f"Pipeline Complete. | "
        f"Rows extracted: {len(df_aligned)} | "
        f"Schema: {list(df_aligned.columns)} | "
        f"Output: {output_file}"
    )

    return str(output_file)

In [ ]:
# ============================================================
# CELL 3 — HTTP EXTRACTION
# ============================================================
def extract_http_features(
    user_input_path: str, 
    start_date: str = None, 
    end_date: str = None, 
    output_prefix: str = "http_features",
    dev_mode: bool = False
) -> str:
    """
    Extracts daily HTTP behavioural features for web activity analysis.
 
    Output schema:
        url_visits               — total URL visits per day
        upload_event_count       — count of upload/post events (NOT bytes)
        keyword_match_indicator  — count of URLs matching 2 hardcoded strings
                                   PRODUCTION: replace with URLhaus / OpenPhish
        unique_external_domains  — distinct URLs visited
        after_hours_browsing     — events outside 06:00–18:00
    """
    logging.info("Initiating HTTP Feature Extraction Pipeline...")
    
    target_file = _validate_extraction_inputs(user_input_path, start_date, end_date, dev_mode)
    output_file = _build_output_path(output_prefix)
    date_filter_sql = _build_date_filter_sql(start_date, end_date)
 
    target_file_str = target_file.as_posix()
    output_file_str = output_file.as_posix()

    # NOTE on `keyword_match_indicator`:
    # This is a TWO-STRING HEURISTIC for demo purposes. It is NOT real
    # threat intelligence. A production system would:
    #   1. Pull URLhaus or OpenPhish daily into a DuckDB table
    #   2. LEFT JOIN that table against http.url
    #   3. Count matches
    # Keeping the heuristic in the demo so the rest of the pipeline still
    # has a domain-reputation signal to fuse on, but the name is honest.
 
    # HTTP-Specific SQL Architecture
    query: str = f"""
    COPY (
        WITH HttpLogs AS (
            SELECT
                user,
                url,
                activity,
                -- CERT http.csv often doesn't have an explicit bytes column, 
                -- so we proxy it via activity or assumed URL depth if needed.
                -- For this schema, we simulate a flag for malicious domains and bytes.
                -- 1. Keep the full timestamp for calculating hours
                strptime(date, '%m/%d/%Y %H:%M:%S') AS raw_timestamp,

                -- 2. Create the truncated date for grouping
                CAST(strptime(date, '%m/%d/%Y %H:%M:%S') AS DATE) AS activity_date
            FROM read_csv('{target_file_str}',columns = {{
                'id':       'VARCHAR',
                'date':     'VARCHAR', 
                'user':     'VARCHAR',
                'url':      'VARCHAR',
                'pc':       'VARCHAR',
                'activity': 'VARCHAR'
            }})
            WHERE user != 'SYSTEM' {date_filter_sql}
        )
        SELECT
            user,
            activity_date,
            
            COUNT(url) AS url_visits,
            
            -- Assuming 'activity' might contain upload indicators in your specific CERT slice
            SUM(CASE WHEN activity ILIKE '%upload%' OR activity ILIKE '%post%' THEN 1 ELSE 0 END) AS upload_event_count,
            
            -- Simple heuristic for malicious domains (in production, you'd join with a Threat Intel feed)
            SUM(CASE WHEN url ILIKE '%wikileaks%' OR url ILIKE '%keylog%' THEN 1 ELSE 0 END) AS keyword_match_indicator,
            
            COUNT(DISTINCT url) AS unique_external_domains,
            
            -- 3. Extract the hour from the raw_timestamp, NOT the activity_date
            SUM(
                CASE WHEN EXTRACT(hour FROM raw_timestamp) < 6 OR EXTRACT(hour FROM raw_timestamp) >= 18 THEN 1 ELSE 0 END
            ) AS after_hours_browsing

        FROM HttpLogs
        GROUP BY user, activity_date
    ) TO '{output_file.as_posix()}' (FORMAT PARQUET);
    """
    # ============================================================
    # 4. EXECUTION BLOCK
    # ============================================================
    try:
        duckdb.sql(query)
    except Exception as e:
        logging.error(f"[extract_http_features] DuckDB execution failed. Reason: {e}")
        raise

    # ============================================================
    # 5. POST-EXECUTION VALIDATION + SCHEMA ALIGNMENT
    # ============================================================
    df_check = pd.read_parquet(output_file)

    if len(df_check) == 0:
        raise RuntimeError("Pipeline produced empty output. Check WHERE filters.")
    
    df_aligned = align_to_master_schema(df_check.set_index(['user', 'activity_date']))
    df_aligned.to_parquet(output_file, index=True)
    
    logging.info(f"HTTP Pipeline Complete. | Rows: {len(df_aligned)} | Output: {output_file}")
    return str(output_file)

In [ ]:
# ============================================================
# CELL 4 — EMAIL EXTRACTION
# ============================================================
def extract_email_features(
    user_input_path: str, 
    start_date: str = None, 
    end_date: str = None, 
    output_prefix: str = "email_features",
    dev_mode: bool = False
) -> str:
    """Extracts daily Email behavioral features for communication analysis."""
    logging.info("Initiating Email Feature Extraction Pipeline...")

    target_file = _validate_extraction_inputs(user_input_path, start_date, end_date, dev_mode)
    output_file = _build_output_path(output_prefix)
    date_filter_sql = _build_date_filter_sql(start_date, end_date)
 
    target_file_str = target_file.as_posix()
    output_file_str = output_file.as_posix()
 
    query: str = f"""
    COPY (
        WITH EmailLogs AS (
            SELECT
                user,
                "to" AS recipient,
                attachment_count,
                size AS email_size,
                -- 1. Keep the full timestamp for calculating hours
                strptime(date, '%m/%d/%Y %H:%M:%S') AS raw_timestamp, 
                
                -- 2. Create the truncated date for grouping
                CAST(strptime(date, '%m/%d/%Y %H:%M:%S') AS DATE) AS activity_date
            FROM read_csv('{target_file_str}',columns = {{
                'id':               'VARCHAR',
                'date':             'VARCHAR',
                'user':             'VARCHAR',
                'pc':               'VARCHAR',
                'to':               'VARCHAR',
                'cc':               'VARCHAR',
                'bcc':              'VARCHAR',
                'from':             'VARCHAR',
                'size':             'VARCHAR',
                'attachment_count': 'VARCHAR',
                'content':          'VARCHAR'           
            }})
            WHERE user != 'SYSTEM' {date_filter_sql}
        )
        SELECT
            user,
            activity_date,
            
            COUNT(*) AS emails_sent,
            
            -- Detect external emails (assuming internal domain is 'dtaa.com' as per CERT standard)
            SUM(CASE WHEN recipient NOT ILIKE '%@dtaa.com%' THEN 1 ELSE 0 END) AS external_recipients,
            
            -- After hours emails (A classic sign of data exfiltration)
            -- 3. Extract the hour from the raw_timestamp, NOT the activity_date
            SUM(
                CASE WHEN EXTRACT(hour FROM raw_timestamp) < 6 OR EXTRACT(hour FROM raw_timestamp) >= 18 THEN 1 ELSE 0 END
            ) AS after_hours_emails,
            
            -- Count attachments by counting semicolons + 1 (if the field isn't empty)
            SUM(
                CASE 
                    WHEN attachment_count IS NOT NULL AND attachment_count != '' 
                    THEN LENGTH(attachment_count) - LENGTH(REPLACE(attachment_count, ';', '')) + 1 
                    ELSE 0 
                END
            ) AS attachments_sent,
            
            -- Flag large attachments (e.g., > 1MB / 1,000,000 bytes)
            SUM(CASE WHEN CAST(email_size AS INTEGER) > 1000000 THEN 1 ELSE 0 END) AS large_attachment_count

        FROM EmailLogs
        GROUP BY user, activity_date
    ) TO '{output_file.as_posix()}' (FORMAT PARQUET);
    """
    # ============================================================
    # 4. EXECUTION BLOCK
    # ============================================================
    try:
        duckdb.sql(query)
    except Exception as e:
        logging.error(f"[extract_email_features] DuckDB execution failed. Reason: {e}")
        raise

    # ============================================================
    # 5. POST-EXECUTION VALIDATION + SCHEMA ALIGNMENT
    # ============================================================
    df_check = pd.read_parquet(output_file)

    if len(df_check) == 0:
        raise RuntimeError("Pipeline produced empty output. Check WHERE filters.")
    
    df_aligned = align_to_master_schema(df_check.set_index(['user', 'activity_date']))
    df_aligned.to_parquet(output_file, index=True)
    
    logging.info(f"Email Pipeline Complete. | Rows: {len(df_aligned)} | Output: {output_file}")
    return str(output_file)     

In [ ]:
# CELL 5: FUSION
# Configure standard logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def fuse_feature_matrices(
    logon_parquet_path: str,
    http_parquet_path: str,
    email_parquet_path: str,
    output_dir: str = "features"
) -> str:
    """
    Fuses separate behavioral feature matrices into a single master dataset.
    (architectural fix):
        Original used `df_logon.combine_first(df_http).combine_first(df_email)`,
        which relies on `align_to_master_schema` having pre-filled missing
        columns with NaN. That was fragile — a future change that imputed
        zeros instead of NaN in any source would silently corrupt the fusion
        (a logon source-zero would override a real HTTP value).
        New behaviour: each source contributes ONLY the columns it owns.
        Outer join on the (user, activity_date) MultiIndex. No ambiguity.
 
    (data-quality fix):
        Removed the `active_mask` filter that dropped rows where all three
        of total_event_buckets/url_visits/emails_sent were zero. Those rows
        are a legitimate IOC ("user disappeared from telemetry") and are
        surfaced downstream by `data_quality_risk` in the detector.
        Filtering them here defeats the security purpose.
    
    """
    logging.info("Initiating Multi-Source Data Fusion...")

    # ============================================================
    # 1. LOAD EXTRACTED DATA
    # ============================================================
    try:
        logging.info("Loading Logon features...")
        df_logon = pd.read_parquet(logon_parquet_path)
        
        logging.info("Loading HTTP features...")
        df_http = pd.read_parquet(http_parquet_path)
        
        logging.info("Loading Email features...")
        df_email = pd.read_parquet(email_parquet_path)
    except FileNotFoundError as e:
        logging.error(f"Missing a required feature file: {e}")
        raise

    # ============================================================
    # 2. VALIDATE INDEX ARCHITECTURE
    # ============================================================
    expected_index = ['user', 'activity_date']
    for name, df in zip(['Logon', 'HTTP', 'Email'], [df_logon, df_http, df_email]):
        if list(df.index.names) != expected_index:
            raise ValueError(f"CRITICAL: {name} DataFrame lacks the required MultiIndex {expected_index}. "
                             f"Current index: {df.index.names}")

    # ============================================================
    # 3. FUSE MATRICES (The Cascade Merge)
    # Each call slices each source to ONLY its own feature columns.
    # `pd.concat(axis=1)` does an outer join on the MultiIndex — any
    # user-day present in ANY source survives, with NaN where a source
    # had no data for them.
    # ============================================================
    logging.info("Fusing matrices via explicit per-source-column concat...")
    
    df_fused = pd.concat(
        [
            df_logon[LOGON_FEATURES],
            df_http[HTTP_FEATURES],
            df_email[EMAIL_FEATURES],
        ],
        axis=1,
        join='outer',
    )
    # ── 4. IMPUTE: NaN → 0 only for known-active user-days ──────────────
    # If a user-day exists in the merged index, they did SOMETHING that
    # day. A NaN in a column means "this source had no events for that
    # user-day" — which is a legitimate zero. The detector's NaN audit
    # treats >20% NaN coverage as an IOC; fillna here means downstream
    # zeros are intentional, not data loss.
    df_fused = df_fused.fillna(0)
   
    # ── 5. (NO active_user filter — removed.) ──────────────────
    # The original filtered out rows where logon/http/email were all
    # zero. That suppressed a real IOC (telemetry blackout). The
    # detector's `data_quality_risk` now surfaces this correctly.
 
    # ============================================================
    # 6. EXPORT
    # ============================================================
    out_path = Path(output_dir).resolve()
    out_path.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = out_path / f"master_features_unscored_{timestamp}.parquet"
    
    df_fused.to_parquet(output_file, index=True)
    
    logging.info(f"Fusion Complete. | Total Unique User-Days: {len(df_fused)}")
    logging.info(f"Master Matrix written to: {output_file}")
    
    return str(output_file)

In [ ]:
# CELL 6: InsiderThreatDetector

class InsiderThreatDetector:
    """
    Dual-model ML pipeline for insider threat detection.
 
    Features:
        • NaN auditing (telemetry gap = IOC)
        • Locked MAD baseline (Model 1)
        • Calibrated Isolation Forest (Model 2)
        • Dual-model fusion with security-hardened threat categories
        • Provenance tracking + integrity-checked deserialization
 
    Why contamination='auto':
        The IsolationForest `contamination` parameter is the assumed
        fraction of anomalies in the training data. The original code
        used `0.05` (5%), which is wildly miscalibrated:
            • CERT r4.2 has ~70 malicious user-days out of ~330,000
              (≈ 0.02%) — 250× lower than 0.05.
            • With contamination=0.05, IsolationForest sets its decision
              threshold so that 5% of training data is "anomalous",
              flooding the SOC with false positives.
        `'auto'` defers to the internal path-length distribution and
        the offset_ attribute, which is statistically defensible without
        requiring us to commit to a specific anomaly rate. The actual
        cut-point is then auto-derived from training scores in
        fit_baseline() and stored as `self.iso_threshold` for inference.
 
    Why MultiIndex (user, activity_date):
        Original `set_index('user')` made each user-day a separate row
        but with a NON-UNIQUE 'user' index — meaning `.loc[user]` could
        return either a row or many rows depending on history, and any
        downstream join silently misbehaved. Using `(user, activity_date)`
        as a MultiIndex makes the unit of analysis explicit (one row =
        one user-day) and keeps the index unique.
    """
    def __init__(self, mad_threshold: float = 3.5, missing_data_tolerance: float = 0.2):
        # Parameterized Thresholds
        self.mad_threshold = mad_threshold
        self.missing_data_tolerance = missing_data_tolerance
        
        # contamination='auto' allows dynamic scoring via decision_function()
        self.iso_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('scaler', RobustScaler()), 
            ('detector', IsolationForest(contamination='auto', random_state=42, n_estimators=100))
        ])
        
        self.baseline_stats = {}
        self.is_fitted = False
        self.model_version = None

    @classmethod
    def load(cls, model_path: str) -> "InsiderThreatDetector":
        """
        Deserializes a saved InsiderThreatDetector from disk.
        Validates structural integrity before returning the object.
    
        Usage:
            detector = InsiderThreatDetector.load("iso_pipeline_v20240101_120000.pkl")
        """
        logging.info(f"[load] Deserializing model from: {model_path}")
    
        try:
            obj = joblib.load(model_path)
        except FileNotFoundError:
            raise FileNotFoundError(
                f"[load] Model file not found at: '{model_path}'. "
                f"Verify the artifact path or rerun fit_baseline()."
        )
        except Exception as e:
            raise RuntimeError(
            f"[load] Failed to deserialize model. "
            f"File may be corrupted or incompatible. Reason: {e}"
        )

        # ── Integrity checks on the loaded object ─────────────────────────────
        if not isinstance(obj, cls):
            raise TypeError(
                f"[load] Loaded object is type '{type(obj).__name__}', "
                f"expected 'InsiderThreatDetector'. Wrong file loaded."
            )
        if not obj.is_fitted:
            raise RuntimeError(
                f"[load] Loaded model has is_fitted=False. "
                f"This model was serialized before training completed."
            )
        if not obj.baseline_stats:
            raise RuntimeError(
            f"[load] Loaded model has empty baseline_stats. "
            f"MAD scoring will be non-functional. Retrain and re-serialize."
        )

        logging.info(
            f"[load] Model loaded successfully | "
            f"version={obj.model_version} | "
            f"features={len(obj.baseline_stats)} | "
            f"iso_threshold={getattr(obj, 'iso_threshold', 'NOT FOUND — retrain recommended')}"
        )
    
        return obj


    def _validate_input(self, df: pd.DataFrame, context: str) -> None:
        """
        Gate-check for all incoming DataFrames.
        Called before any model training or inference begins.
        Raises ValueError with actionable messages on any violation.
        """

        # ── CHECK 1: Empty DataFrame ──────────────────────────────────────────
        if len(df) == 0:
            raise ValueError(
                f"[{context}] Empty DataFrame received. "
                f"Check if the upstream ETL pipeline produced output."
            )

        # ── CHECK 2: 'user' must exist somewhere (column or index) ─────────────────────────────────
        has_user_col   = 'user' in df.columns
        has_user_index = 'user' in (df.index.names or [])
        if not (has_user_col or has_user_index):
            raise ValueError(
                f"[{context}] Required identifier 'user' is missing from "
                f"both columns and index. "
                f"Columns: {list(df.columns)} | Index: {df.index.names}"
            )

        # ── CHECK 3: Duplicate Time-Series Records ────────────────────────────
        # Since we upgraded to a time-series architecture, users will have multiple rows.
        # We must check if they have duplicate rows for the SAME DAY.
        if 'activity_date' in df.columns and has_user_col:
            if df.duplicated(subset=['user', 'activity_date']).any():
                raise ValueError(
                    f"[{context}] Duplicate User+Date entries detected. "
                    f"A user cannot have two baseline records for the exact same day. "
                    f"Check upstream SQL GROUP BY logic."
                )
        
        # Numeric columns must exist
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) == 0:
            raise ValueError(
                f"[{context}] No numeric feature columns found. "
                f"Columns: {list(df.columns)}"
            )    

        # Feature drift (inference only)
        if self.is_fitted and context == "predict_live_traffic":
            trained = set(self.baseline_stats.keys())
            live    = set(numeric_cols)
            missing = trained - live
            new     = live - trained
 
            if missing:
                raise ValueError(
                    f"[{context}] FEATURE DRIFT — trained features missing from "
                    f"live data: {missing}. Retrain or fix ETL."
                )
            if new:
                logging.warning(
                    f"[{context}] New unseen columns in live data: {new}. "
                    "These will be ignored by the model."
                )
 
        logging.info(
            f"[{context}] Input validation passed. "
            f"Rows: {len(df)} | Numeric features: {len(numeric_cols)}"
        )
    # ────────────────────────────────────────────────────────────────────
    # Helper: load parquet and put it on a (user, activity_date) MultiIndex
    # replaces the old `df_raw.set_index('user')` everywhere.
    # ────────────────────────────────────────────────────────────────────
    @staticmethod
    def _load_with_user_day_index(parquet_path: str, context: str) -> pd.DataFrame:
        """
        Loads parquet and returns a DataFrame indexed by ('user', 'activity_date').
        Handles both cases:
            • parquet already has the MultiIndex (from fuse_feature_matrices)
            • parquet has 'user' and 'activity_date' as plain columns
        Raises FileNotFoundError / RuntimeError with actionable messages.
        """
        try:
            df = pd.read_parquet(parquet_path)
        except FileNotFoundError:
            raise FileNotFoundError(
                f"[{context}] Parquet file not found at: '{parquet_path}'. "
                f"Verify the path and check the upstream ETL job."
            )
        except Exception as e:
            raise RuntimeError(f"[{context}] Failed to load parquet. Reason: {e}")
 
        # Case 1: MultiIndex already correct
        if list(df.index.names) == ['user', 'activity_date']:
            return df
 
        # Case 2: index needs rebuilding from columns
        if 'user' in df.index.names:
            df = df.reset_index()
        if 'user' in df.columns and 'activity_date' in df.columns:
            return df.set_index(['user', 'activity_date'])
 
        raise ValueError(
            f"[{context}] Cannot construct (user, activity_date) MultiIndex. "
            f"Found columns: {list(df.columns)}, index: {df.index.names}"
        )  
    # ────────────────────────────────────────────────────────────────────
    # TRAINING
    # ────────────────────────────────────────────────────────────────────
    def fit_baseline(self, historical_parquet_path: str):
        """Trains the model on known-clean historical data and exports the serialized model."""
        logging.info(f"TRAINING PHASE: Loading historical baseline from {historical_parquet_path}")
        # use the helper, which gives us a (user, activity_date) MultiIndex
        df_features = self._load_with_user_day_index(
            historical_parquet_path, context="fit_baseline"
        )
 
        # Validate before any model touches the data
        # We pass a temporary reset-index view so column-based checks work
        df_for_validation = df_features.reset_index()
        self._validate_input(df_for_validation, context="fit_baseline")
    
        # ── 1. MAD Baseline (Model 1) ───────────────────────────────────
        numeric_cols = df_features.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            col_median = df_features[col].median(skipna=True)
            col_mad    = median_abs_deviation(df_features[col].dropna(), scale='normal')
            self.baseline_stats[col] = {'median': col_median, 'mad': col_mad}
 
        zero_mad_cols = [c for c, v in self.baseline_stats.items() if v['mad'] == 0]
        logging.info(
            f"[fit_baseline] MAD baseline computed | "
            f"features={len(self.baseline_stats)} | zero_mad_cols={zero_mad_cols}"
        )
         
        # ── 2. Isolation Forest (Model 2) ───────────────────────────────
        self.iso_pipeline.fit(df_features[numeric_cols])
        self.is_fitted = True
 
        # Auto-derive ISO threshold from training distribution
        train_scores = self.iso_pipeline.decision_function(df_features[numeric_cols])
        self.iso_threshold = float(np.mean(train_scores) - np.std(train_scores))
        logging.info(
            f"[fit_baseline] ISO threshold auto-derived | "
            f"mean={np.mean(train_scores):.4f} | std={np.std(train_scores):.4f} | "
            f"threshold={self.iso_threshold:.4f}"
        )
 
        # store training score percentiles for inference calibration ──
        quantile_points = np.linspace(0, 1, 1000)
        self.train_score_percentiles = list(
            np.quantile(train_scores, quantile_points).astype(float)
        )
        logging.info(
            f"[fit_baseline] Training score percentiles stored | "
            f"min={min(self.train_score_percentiles):.4f} | "
            f"p50={self.train_score_percentiles[500]:.4f} | "
            f"max={max(self.train_score_percentiles):.4f}"
        )
        
        # Model Serialization & Provenance
        self.model_version = datetime.now().strftime("%Y%m%d_%H%M%S")
        export_path = f"iso_pipeline_v{self.model_version}.pkl"
        joblib.dump(self, export_path)
        logging.info(
            f"[fit_baseline] Model serialized | version={self.model_version} | "
            f"path={export_path} | features={list(self.baseline_stats.keys())}"
        )

        return export_path
    # ────────────────────────────────────────────────────────────────────
    # INFERENCE
    # ────────────────────────────────────────────────────────────────────    
    def predict_live_traffic(self, live_parquet_path: str, iso_decision_threshold: float = None) -> pd.DataFrame:
        """
        Scores live traffic against the locked historical baseline.
 
        Returns a DataFrame indexed by (user, activity_date) containing:
            data_quality_risk     — 1 if >20% of features are NaN
            mad_score_count       — feature count exceeding MAD threshold
            mad_critical_flag     — 1 if mad_score_count >= 2
            iso_forest_raw_score  — continuous ISO Forest decision_function
            iso_forest_flag       — 1 if score < threshold
            confirmed_threat      — both models + clean data
            high_risk_review      — both models + dirty data
            data_loss_ioc         — telemetry gap only (no model signal)
 
        output is now indexed by (user, activity_date) — unique rows.
        """
        if not self.is_fitted:
            raise RuntimeError(
                "Model must be fit() before predict() is called."
                 "Run fit_baseline() first."  
            )
        # ── Resolve threshold: caller override → auto-derived → hard fallback ─
        if iso_decision_threshold is None:
            if hasattr(self, 'iso_threshold'):
                iso_decision_threshold = self.iso_threshold
                logging.info(f"[predict_live_traffic] Using auto-derived threshold: {iso_decision_threshold:.4f}")
            else:
                iso_decision_threshold = -0.1  # last-resort fallback, logged clearly
                logging.warning(
                    "[predict_live_traffic] iso_threshold not found on model. "
                    "Falling back to default -0.1. Retrain model to fix this."
                )
        # MultiIndex (user, activity_date)
        df_features = self._load_with_user_day_index(
            live_parquet_path, context="predict_live_traffic"
        )
 
        # Validate (reset-index view for column-based checks)
        self._validate_input(df_features.reset_index(), context="predict_live_traffic")
 
        logging.info(f"INFERENCE PHASE: Evaluating live traffic from {live_parquet_path}")
 
        # Scores DataFrame inherits the (user, activity_date) MultiIndex
        df_scores = pd.DataFrame(index=df_features.index)
 
        # ── NaN Audit (data_quality_risk IOC) ───────────────────────────
        missing_pct = df_features.isnull().mean(axis=1)
        df_scores['data_quality_risk'] = np.where(
            missing_pct > self.missing_data_tolerance, 1, 0
        )
        if df_scores['data_quality_risk'].sum() > 0:
            logging.warning(
                f"BLIND SPOT ALERT: {df_scores['data_quality_risk'].sum()} "
                f"user-days flagged for suspicious data loss."
            )
 
        # ── MODEL 1: MAD against locked baseline ────────────────────────
        mad_flags_matrix = pd.DataFrame(index=df_features.index)
        df_for_mad = df_features.fillna(0)
 
        for col in self.baseline_stats.keys():
            if col not in df_for_mad.columns:
                continue
            hist_median = self.baseline_stats[col]['median']
            hist_mad    = self.baseline_stats[col]['mad']
            live_values = df_for_mad[col]
 
            if hist_mad == 0:
                # Iglewicz-Hoaglin can't compute z when MAD=0; we still
                # want to flag values strictly above the training median.
                # Constant 10.0 is a sentinel z-score that exceeds any
                # realistic mad_threshold (default 3.5).
                z_scores = np.where(live_values > hist_median, 10.0, 0.0)
            else:
                # Modified z-score (Iglewicz-Hoaglin): 0.6745 = phi^-1(0.75)
                z_scores = np.abs(0.6745 * (live_values - hist_median) / hist_mad)
 
            mad_flags_matrix[f'{col}_flag'] = np.where(z_scores > self.mad_threshold, 1, 0)
 
        df_scores['mad_score_count']   = mad_flags_matrix.sum(axis=1)
        df_scores['mad_critical_flag'] = np.where(df_scores['mad_score_count'] >= 2, 1, 0)
        logging.info(
            f"[predict_live_traffic] MAD scoring complete | "
            f"critical_flags={df_scores['mad_critical_flag'].sum()} | "
            f"avg_flag_count={df_scores['mad_score_count'].mean():.2f}"
        )
 
        # ── MODEL 2: Isolation Forest ───────────────────────────────────
        numeric_cols = df_features.select_dtypes(include=[np.number]).columns
        df_scores['iso_forest_raw_score'] = self.iso_pipeline.decision_function(
            df_features[numeric_cols]
        )
        df_scores['iso_forest_flag'] = np.where(
            df_scores['iso_forest_raw_score'] < iso_decision_threshold, 1, 0
        )
        logging.info(
            f"[predict_live_traffic] ISO Forest scoring complete | "
            f"range=[{df_scores['iso_forest_raw_score'].min():.4f}, "
            f"{df_scores['iso_forest_raw_score'].max():.4f}] | "
            f"threshold={iso_decision_threshold:.4f} | "
            f"flagged={df_scores['iso_forest_flag'].sum()}"
        )
 
        # ── DATA FUSION ─────────────────────────────────────────────────
        df_scores['confirmed_threat'] = np.where(
            (df_scores['mad_critical_flag'] == 1) &
            (df_scores['iso_forest_flag']   == 1) &
            (df_scores['data_quality_risk'] == 0),
            1, 0
        )
        df_scores['high_risk_review'] = np.where(
            (df_scores['mad_critical_flag'] == 1) &
            (df_scores['iso_forest_flag']   == 1) &
            (df_scores['data_quality_risk'] == 1),
            1, 0
        )
        df_scores['data_loss_ioc'] = np.where(
            (df_scores['data_quality_risk'] == 1) &
            (df_scores['confirmed_threat']  == 0) &
            (df_scores['high_risk_review']  == 0),
            1, 0
        )
        logging.info(
            f"[predict_live_traffic] Threat summary | "
            f"confirmed={df_scores['confirmed_threat'].sum()} | "
            f"high_risk_review={df_scores['high_risk_review'].sum()} | "
            f"data_loss_ioc={df_scores['data_loss_ioc'].sum()}"
        )
 
        return df_scores
        

In [ ]:
# CELL 7: EXECUTION & EXTRACTION

def run_extraction_phase():
    """Runs all three extractions for both historical and live windows."""
    # 1. Historical training data
    print("Extracting Historical Baseline...")
    hist_logon = extract_logon_features(
        user_input_path="data/r4.2/logon.csv",
        start_date="2010-01-01", end_date="2010-07-01", dev_mode=True,
    )
    hist_http = extract_http_features(
        user_input_path="data/r4.2/http.csv",
        start_date="2010-01-01", end_date="2010-07-01", dev_mode=True,
    )
    hist_email = extract_email_features(
        user_input_path="data/r4.2/email.csv",
        start_date="2010-01-01", end_date="2010-07-01", dev_mode=True,
    )
 
    # 2. Live test data
    print("Extracting Live Test Traffic...")
    live_logon = extract_logon_features(
        user_input_path="data/r4.2/logon.csv",
        start_date="2011-04-01", end_date=None, dev_mode=True,
    )
    live_http = extract_http_features(
        user_input_path="data/r4.2/http.csv",
        start_date="2011-04-01", end_date=None, dev_mode=True,
    )
    live_email = extract_email_features(
        user_input_path="data/r4.2/email.csv",
        start_date="2011-04-01", end_date=None, dev_mode=True,
    )
 
    return {
        'hist_logon': hist_logon, 'hist_http': hist_http, 'hist_email': hist_email,
        'live_logon': live_logon, 'live_http': live_http, 'live_email': live_email,
    }


In [ ]:
# ==============================================================
# CELL 8: FUSING ALL PARQUET FILES
# ==============================================================
def run_fusion_phase(extracted_paths: dict) -> tuple[str, str]:
    """Fuses the three sources for both windows. Returns (historical, live) paths."""
    fused_historical = fuse_feature_matrices(
        logon_parquet_path=extracted_paths['hist_logon'],
        http_parquet_path =extracted_paths['hist_http'],
        email_parquet_path=extracted_paths['hist_email'],
    )
    fused_live = fuse_feature_matrices(
        logon_parquet_path=extracted_paths['live_logon'],
        http_parquet_path =extracted_paths['live_http'],
        email_parquet_path=extracted_paths['live_email'],
    )
    return fused_historical, fused_live

In [ ]:
# CELL 9 : TRAIN + PREDICT EXECUTION

def run_modeling_phase(fused_historical_path: str, fused_live_path: str):
    """Trains the detector on history and scores the live window."""
    detector = InsiderThreatDetector(mad_threshold=3.5, missing_data_tolerance=0.2)
 
    print("Training the ML model on historical baseline...")
    detector.fit_baseline(fused_historical_path)
 
    print("Scoring the live test data...")
    results_df = detector.predict_live_traffic(fused_live_path)
    return detector, results_df


In [ ]:
# MAIN ENTRY POINT
# Mirrors the notebook execution order (Cell 7 → Cell 8 → Cell 9)
if __name__ == "__main__":
    extracted = run_extraction_phase()
    fused_hist, fused_live = run_fusion_phase(extracted)
    detector, results = run_modeling_phase(fused_hist, fused_live)
 
    print("\nPipeline complete.")
    print(f"  Model version : {detector.model_version}")
    print(f"  Rows scored   : {len(results)}")
    print(f"  Confirmed     : {int(results['confirmed_threat'].sum())}")
    print(f"  Review        : {int(results['high_risk_review'].sum())}")
    print(f"  Data loss IOC : {int(results['data_loss_ioc'].sum())}")
